In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import difflib
import re

In [7]:
bandages = pd.read_excel("../../data/cleaned/acceptances/bandages.xlsx")
runners = pd.read_excel("../../data/cleaned/results/runners.xlsx")

In [5]:
# 1. Get unique names and clean them to find potential duplicates
unique_names = bandages.horse_name.dropna().unique()

# 2. Look for close matches
similar_pairs = []
for i, name1 in enumerate(unique_names):
    # Get close matches from the rest of the list
    matches = difflib.get_close_matches(name1, unique_names[i+1:], n=3, cutoff=0.8)
    for match in matches:
        similar_pairs.append((name1, match))

# 3. View the results clearly
df_typos = pd.DataFrame(similar_pairs, columns=['Name A', 'Name B'])
df_typos

,Name A,Name B
0,SPARKLING THEA,SPARKLING DEW
1,RARE SILVER,STAR SILVER
2,ADELE,ADELINE
3,LUCAS,LUCA
4,GOLDEN TOWER,GOLDEN ERA
5,DIVINE SPARK,DIVINE STAR
6,GOLDEN EVE,GOLDEN ERA
7,ZANARA,ANAIRA
8,MARIANELLA,MARIELLA
9,WESTERN STYLE,WESTERN STAR


In [ ]:
unique_names = bandages.horse_name.dropna().unique()

# Create a DataFrame of unique names
df_names = pd.DataFrame({'original_name': unique_names})

# Create a 'clean' key: uppercase, no spaces, no punctuation
df_names['clean_key'] = df_names['original_name'].apply(
    lambda x: re.sub(r'[^A-Z0-9]', '', str(x).upper())
)

# Find clean keys that appear more than once (meaning they have variations)
duplicates = df_names[df_names.duplicated(subset=['clean_key'], keep=False)]

# Sort them so variations sit next to each other
duplicates.sort_values(by='clean_key')

,original_name,clean_key


In [ ]:
# =========================
# 2. NORMALIZE
# =========================
def normalize(df):
    df = df.copy()
    df['meet_date'] = pd.to_datetime(df['meet_date'])
    df['horse_name'] = df['horse_name'].str.strip().str.upper()
    return df

runners = normalize(runners)
bandages = normalize(bandages)

# =========================
# 3. DETECT DUPLICATES IN RUNNERS (RISK)
# =========================
dup_check = runners.groupby(['meet_date', 'horse_name']).size().reset_index(name='count')
duplicate_horses = dup_check[dup_check['count'] > 1]

print("Potential ambiguous horses:", len(duplicate_horses))
display(duplicate_horses.head())

# =========================
# 4. BUILD MAPPING (GROUND TRUTH)
# =========================
runner_map = runners[
    ['meet_date', 'horse_name', 'race_no']
].drop_duplicates()

# =========================
# 5. MERGE
# =========================
merged = bandages.merge(
    runner_map,
    on=['meet_date', 'horse_name'],
    how='left',
    indicator=True,
    suffixes=('_old', '_true')
)

# =========================
# 6. KEEP VALID + FIX RACE_NO
# =========================
clean_bandages = merged[merged['_merge'] == 'both'].copy()

clean_bandages['race_no'] = clean_bandages['race_no_true']

clean_bandages = clean_bandages.drop(
    columns=['_merge', 'race_no_old', 'race_no_true']
)

# =========================
# 7. EDGE CASES (NOT FOUND IN RUNNERS)
# =========================
edge_cases = merged[merged['_merge'] == 'left_only'].copy()

edge_case_list = edge_cases[
    ['meet_date', 'horse_name']
].drop_duplicates()

print("Dropped (non-runners):", len(edge_case_list))
display(edge_case_list.head())

# =========================
# 8. FINAL SORT
# =========================
clean_bandages = clean_bandages.sort_values(
    by=['meet_date', 'race_no']
).reset_index(drop=True)

# Convert to string date format right before saving
clean_bandages['meet_date'] = clean_bandages['meet_date'].dt.strftime('%Y-%m-%d')


# =========================
# 9. SAVE
# =========================
clean_bandages.to_excel("../../data/cleaned/acceptances_cleaned/bandages.xlsx", index=False)

Potential ambiguous horses: 0


,meet_date,horse_name,count


Dropped (non-runners): 155


,meet_date,horse_name
540,2018-09-16,JAGER BOMB
937,2019-01-24,JAGER BOMB
1104,2019-03-17,MARIANELLA
1336,2019-09-07,JACK FLASH
1338,2019-09-07,NIGHTFALL


In [ ]:
# 1. Map out where the race number changes from the row above it
consecutive_groups = (bandages['race_no'] != bandages['race_no'].shift()).cumsum().values

# 2. Group by the array, the meet_date, and the race column
# This pulls meet_date into the resulting aggregation
race_counts = bandages.groupby([consecutive_groups, 'meet_date', 'race_no']).size().reset_index()

# 3. Rename columns cleanly to match the new structure
race_counts.columns = ['group_id', 'meet_date', 'race_no', 'count']

# 4. Filter for counts greater than 19 and display specific columns
filtered_counts = race_counts[race_counts['count'] > 19][['meet_date', 'race_no', 'count']]

with pd.option_context('display.max_rows', None):
    display(filtered_counts)

,meet_date,race_no,count
434,2011-02-06,184,22
437,2011-02-06,187,22
505,2011-03-06,255,22
668,2011-08-07,47,21
889,2011-11-27,34,20
1031,2012-02-04,176,22
1036,2012-02-04,181,20
1071,2012-02-19,216,20
1664,2013-02-02,171,20
1665,2013-02-02,172,20


In [ ]:
output_file = "../data/cleaned/acceptances_cleaned/acceptances.xlsx"

runners.to_excel(output_file, index=False)